In [0]:
spark.conf.set(
    "fs.azure.account.key.stdeltalakedemo.dfs.core.windows.net",
    "00000000000000000000000000000000000000000000000000000000000000000000000000000000000000==")

In [0]:
from pyspark.sql.functions import col, sum
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException

def is_valid_delta_table(path: str) -> bool:
    try:
        # Ensure directory exists
        entries = dbutils.fs.ls(path)

        # Explicitly check for _delta_log directory
        has_delta_log = any(f.name.rstrip("/") == "_delta_log" for f in entries)

        if not has_delta_log:
            return False

        # Validate Delta metadata
        return DeltaTable.isDeltaTable(spark, path)

    except AnalysisException:
        return False
    except FileNotFoundError:
        return False

# Define widgets (ADF will inject values here)
dbutils.widgets.text("p_trainee_id", "")
dbutils.widgets.text("p_silver_path", "")
dbutils.widgets.text("p_gold_path", "")

# Retrieve parameter values
p_trainee_id = dbutils.widgets.get("p_trainee_id")
silver_path  = dbutils.widgets.get("p_silver_path")
gold_path    = dbutils.widgets.get("p_gold_path")

# Read silver layer
silver_df = spark.read.format("delta").load(silver_path)

# Aggregate to gold layer
silver_df = silver_df.withColumn(
"amount",
col("amount").cast("double")
)

# Aggregate:
gold_df = (
silver_df.groupBy("status")
.agg(sum("amount").alias("total_amount"))
)
# Merge into gold
if is_valid_delta_table(gold_path):
    gold_delta = DeltaTable.forPath(spark, gold_path)
    (
        gold_delta.alias("gold")
        .merge(
            gold_df.alias("updates"),
            "gold.status = updates.status"
        )
        .whenNotMatchedInsert(
            values={"status": col("updates.status"), "total_amount": col("updates.total_amount")}
        )
        .execute()
    )
else:
    gold_df.write.format("delta").mode("overwrite").save(gold_path)

---------------------------------------------------------------------------
IllegalArgumentException                  Traceback (most recent call last)
File <command-4949399039577633>, line 35
     33 # Validate schema before aggregation
     34 required_cols = {"status", "amount"}
---> 35 missing_cols = required_cols - set(silver_df.columns)
     36 if missing_cols:
     37     raise ValueError(f"Missing required columns in silver_df: {missing_cols}")

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:309, in DataFrame.columns(self)
    307 @property
    308 def columns(self) -> List[str]:
--> 309     return [field.name for field in self._schema.fields]

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1970, in DataFrame._schema(self)
   1968 if self._cached_schema is None:
   1969     query = self._plan.to_proto(self._session.client)
-> 1970     self._cached_schema = self._session.client.schema(query)
   1971     try:
   1972         self._cached_schema_ser